<a href="https://colab.research.google.com/github/UPEI-CS/CS1910-Students/blob/main/lectures/10-Problem-Solving-Debugging/CS1910_Problem-Solving-Debugging-Ants_Handout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CS1910 Computer Science 1: Problem Solving**
# **Ants on a Stick**

## Save your own copy first

You will probably want your own copy of this workbook, so that you can keep your answers and come back to review it. **Do this before you start**, because anything you type into the original is not saved. Choose:

***File->Save a copy in Drive***

or

***Copy to Drive*** if it appears in the above menu.

Google sign-in is required. You will then be able to open the copy from your Google Drive account later (Colab keeps it in the *Colab Notebooks* folder).

If you can't use Google Drive, choose ***File->Download .ipynb*** and keep the file by email (signing in to a cloud email service) or on a USB key. To open it in another Colab session later, you would have to move it into your Google Drive account.

Remember that you can always open the original version from the course Moodle. It just won't contain anything you have typed into its cells.

## How to use this notebook

Start this in lecture **with a partner**, and finish it at home. It takes about 60 minutes.

This problem is about the **process** of problem solving, not just the final code. Work through the four stages in order: **Understand → Design → Implement → Evaluate**. Resist the urge to jump straight to code!

1. **Run the setup cell(s)** (▶️ below) first.
2. Answer the questions in the ✏️ cells (double-click a ✏️ cell to edit it).
3. When a code cell ends with checks, keep working until every check shows ✅.

If you are stuck for more than 10 minutes, write down what you tried and move on, then bring your question to office hours or the help centre.

## What this notebook practises

A puzzle that looks like it needs a complicated simulation turns out to have a short, elegant answer, *if* you follow the process. You will pin down exactly what the question is asking, explore small cases with a simulator, spot the pattern that makes collisions irrelevant, and then test your fast solution against a slow one that you know is right.

### Key points to take away

- **Understand** before you design: find what is ambiguous (how long is the stick? which way are the ants facing?) and pin it down.
- Work **small cases** by hand or with a simulator before you look for a general rule.
- **Abstraction:** two ants bouncing off each other look exactly like two ants walking *through* each other. Labels aside, the picture is the same, so collisions can be ignored.
- Then each ant is independent: it falls off after `p` seconds (going left) or `10 - p` seconds (going right). The stick is empty when the **last** ant falls.
- Earliest time = the largest of each ant's *nearer*-end time. Latest time = the largest of each ant's *farther*-end time.
- **Evaluate** a clever solution by comparing it with a slow, simple one (a *brute-force* simulation) on many inputs.

In [ ]:
#@title ▶️ Run this cell first
#@markdown This cell sets up two helpers, `check` and `check_case`, that tell you
#@markdown whether your answer is right. You do not need to understand how they work.
import math as _math

_cases_passed = {}


def _same(got, expected):
    if isinstance(expected, float) and isinstance(got, (int, float)) and not isinstance(got, bool):
        return _math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
    return type(got) == type(expected) and got == expected


def check(got, expected, label="Answer"):
    if _same(got, expected):
        print(f"✅ {label}: {got!r}")
    elif got == expected:
        print(f"❌ {label}: the value looks right, but it is a {type(got).__name__}"
              f" and should be a {type(expected).__name__}")
    else:
        print(f"❌ {label}: got {got!r}, expected {expected!r}")


def check_case(name, case, got, expected_by_case):
    if case not in expected_by_case:
        print(f"ℹ️ {case!r} is not one of the test cases for {name}. Try one of: "
              + ", ".join(repr(c) for c in expected_by_case))
        return
    passed = _cases_passed.setdefault(name, set())
    check(got, expected_by_case[case], f"{name} with {case!r}")
    if _same(got, expected_by_case[case]):
        passed.add(case)
    still = [repr(c) for c in expected_by_case if c not in passed]
    if still:
        print(f"   {len(passed)} of {len(expected_by_case)} test cases passed. "
              f"Change the input and run again. Still to try: {', '.join(still)}")
    else:
        print(f"   🎉 All {len(expected_by_case)} test cases passed!")


print("✅ Practice checker ready.")

In [ ]:
#@title 🐜 Run this cell second: the ant simulator
#@markdown This cell gives you three helpers. You do not need to understand how they work.
#@markdown - `show_ants(positions, directions)` draws the stick every half second and returns the time it is empty.
#@markdown - `compare_ants(positions, directions)` draws two sticks side by side: ants that **bounce** (left) and ants that **pass through** each other (right).
#@markdown - `simulate_all(positions)` tries **every** possible choice of directions and prints the earliest and latest times.

LENGTH = 10   # the stick is 10 cm long


def _run(positions, directions, bounce):
    """Simulate in half-second steps. Returns (rows, time): one drawing per half second."""
    if len(positions) != len(directions):
        raise ValueError("give one direction (L or R) for each ant")
    ants = []
    for k in range(len(positions)):
        ants.append([2 * int(positions[k]), -1 if directions[k].upper() == "L" else 1, "ABCDEFGHI"[k]])
    rows = []
    t = 0
    while True:
        cells = ["-"] * (2 * LENGTH + 1)
        for a in ants:
            cells[a[0]] = "X" if cells[a[0]] not in "-" else a[2]
        rows.append(f"t = {t / 2:4.1f} s  |" + "".join(cells) + "|")
        if not ants:
            return rows, t / 2
        t += 1
        for a in ants:
            a[0] += a[1]
        ants = [a for a in ants if 0 < a[0] < 2 * LENGTH]
        if bounce:
            for i in range(len(ants)):
                for j in range(i + 1, len(ants)):
                    if ants[i][0] == ants[j][0] and ants[i][1] != ants[j][1]:
                        ants[i][1] = -ants[i][1]
                        ants[j][1] = -ants[j][1]


def show_ants(positions, directions):
    rows, time = _run(positions, directions, True)
    print(" " * 13 + "0" + " " * 9 + "5" + " " * 9 + "10 cm")
    for r in rows:
        print(r)
    print(f"The stick is empty after {time} seconds.")
    return time


def compare_ants(positions, directions):
    left, t1 = _run(positions, directions, True)
    right, t2 = _run(positions, directions, False)
    print(f"{'ants BOUNCE':<35}ants PASS THROUGH")
    for k in range(max(len(left), len(right))):
        a = left[k] if k < len(left) else ""
        b = right[k][11:] if k < len(right) else ""
        print(f"{a:<35}{b}")
    print(f"Empty after {t1} s (bounce) and {t2} s (pass through).")


def simulate_all(positions):
    """Try all 2**n direction choices. Returns nothing; prints the earliest and latest times."""
    n = len(positions)
    times = []
    for code in range(2 ** n):
        directions = ""
        for k in range(n):
            directions += "R" if (code >> k) & 1 else "L"
        times.append(_run(positions, directions, True)[1])
    print(f"{2 ** n} direction choices simulated for ants at {positions!r}: "
          f"earliest {min(times)} s, latest {max(times)} s")


def sim_earliest(positions):
    n = len(positions)
    return min(_run(positions, "".join("R" if (c >> k) & 1 else "L" for k in range(n)), True)[1] for c in range(2 ** n))


def sim_latest(positions):
    n = len(positions)
    return max(_run(positions, "".join("R" if (c >> k) & 1 else "L" for k in range(n)), True)[1] for c in range(2 ** n))


print("🐜 Ant simulator ready.")

## Stage 1: Understand (about 15 minutes)


Here is the problem as it was first given to you:

> Ants march on a stick. They travel in one direction, but if they bump into another ant both ants reverse direction and walk the other way. If they reach the end of the stick they fall off. Ants travel at 1 cm/s. What is the minimum or maximum amount of time until the stick has no ants on it?
>
> *Input:* the number of ants on the stick and the starting location of each ant.
> *Output:* the minimum and maximum possible time for the stick to empty.

Before designing anything, find out what you **don't** know.

Answer these with your partner.

1. What are the inputs, and what is the output? What types might they be?
2. The output is the minimum and maximum **possible** time. Possible over *what*? What piece of information about each ant is deliberately missing?
3. List at least **four** questions you would ask before you start designing. (Think about the stick, the ants' positions, collisions, and unusual inputs.)
4. Which details in the story can be ignored completely? (Abstraction: what does not affect the answer?)

✏️ **Your answers** (double-click to edit):

1. 
2. 
3. 
4. 

### The clarified problem

After asking questions, here is the precise problem we will solve:

> - The stick is **10 cm** long. Positions are measured in cm from the left end, so the ends are at **0** and **10**.
> - Each ant starts at a different **whole-number** position from **1 to 9**.
> - Each ant starts facing **left or right**, but **we do not know which**.
> - Every ant walks at **1 cm per second**. When two ants meet, both turn around instantly.
> - An ant falls off the moment it reaches either end.
> - **Input:** a string with one digit per ant, giving its starting position. For example `"28"` means two ants, at 2 cm and 8 cm.
> - **Output:** the **earliest** and the **latest** time (in seconds) that the stick could become empty, over every possible choice of starting directions. With no ants (`""`), both are 0.

### Simple cases first

With only one ant there are no collisions. Fill in the values.

In [ ]:
# 1.1 Work it out by hand: ONE ant at 3 cm.
# How long until it falls off if it walks left? If it walks right?
# What are the earliest and latest times the stick could be empty?
ant3_left = 0
ant3_right = 0
ant3_earliest = 0
ant3_latest = 0

# ---- Checks: run the cell to test your answers (do not change below) ----
check(ant3_left, 3, "ant3_left")
check(ant3_right, 7, "ant3_right")
check(ant3_earliest, 3, "ant3_earliest")
check(ant3_latest, 7, "ant3_latest")

## Stage 2: Explore with a simulator (about 10 minutes)


Two ants are where it gets hard to do by hand, because they might collide. Instead of guessing, use the simulator. `show_ants(positions, directions)` draws the stick every half second (each `-` is half a centimetre) and tells you when it is empty. `X` marks two ants meeting.

Run the cell, then change the directions to try all four possibilities for two ants at 2 cm and 8 cm: `"LL"`, `"LR"`, `"RL"` and `"RR"`.

In [ ]:
show_ants("28", "RL")

### Record what you see

✏️ **Your table** (double-click to edit):

| Directions | Does a collision happen? | Time until the stick is empty |
|---|---|---|
| `"LL"` | | |
| `"LR"` | | |
| `"RL"` | | |
| `"RR"` | | |

In [ ]:
simulate_all("28")

1. What are the earliest and latest times for `"28"`? Which choices of direction give each one?
2. Try `show_ants("34", "RL")`: the ants start 1 cm apart, facing each other. Where and when do they meet?
3. Use `simulate_all` on two more inputs of your choice. Write down the inputs and results.

✏️ **Your answers** (double-click to edit):

1. 
2. 
3. 

## Stage 3: Recognize the pattern (about 10 minutes)


`compare_ants(positions, directions)` runs the same ants twice, side by side: on the left the ants **bounce** off each other as in the problem; on the right they **pass through** each other like ghosts. Run it, then try `compare_ants("34", "RL")` and `compare_ants("2468", "RLRL")`.

In [ ]:
compare_ants("28", "RL")

Answer these with your partner.

1. Ignore the letters and look only at **where** the marks are on each row. What do you notice about the two pictures?
2. So what is the only difference between bouncing and passing through? Does it change *when* the stick becomes empty?
3. If ants pass through each other, how long does an ant at position `p` take to fall off if it walks left? If it walks right?
4. The stick is empty when the **last** ant falls off. Write a rule for the time the stick becomes empty, for one particular choice of directions.
5. To make the stick empty as **early** as possible, which way should each ant walk? Write a formula for the earliest time.
6. To make it empty as **late** as possible, what is the best choice? Write a formula for the latest time.
7. Your friend says: "the earliest time is the *smallest* nearer-end distance of any ant". Test the claim on `"15"`. Are they right?

✏️ **Your answers** (double-click to edit):

1. 
2. 
3. 
4. 
5. 
6. 
7. 

> 🧠 **Key ideas: the ants**
>
> - Bouncing ants and passing-through ants produce the **same picture**, so collisions never change when the stick is empty.
> - Each ant then takes `p` seconds (left) or `10 - p` seconds (right).
> - **Earliest** time = the largest of `min(p, 10 - p)` over all ants.
> - **Latest** time = the largest of `max(p, 10 - p)` over all ants.

## Stage 4: Design (about 5 minutes)

Turn the pattern into an algorithm. You will write two functions, `earliest_empty(positions)` and `latest_empty(positions)`, each taking the input string and returning a whole number of seconds.

Write your plan for both functions as pseudocode. Check it against `"28"`, `"15"` and `""` before you code.

✏️ **Your plan** (double-click to edit; write it before you code):

1. 
2. 
3. 

## Stage 5: Implement (about 10 minutes)

Turn your plan into Python. You may use `min(a, b)` and `max(a, b)`, or `if` statements.

In [ ]:
def earliest_empty(positions):
    # Your code here (follow your plan)
    pass


def latest_empty(positions):
    # Your code here (follow your plan)
    pass

# ---- Checks: run the cell to test your code (do not change below) ----
check(earliest_empty("3"), 3, 'earliest_empty("3")')
check(latest_empty("3"), 7, 'latest_empty("3")')
check(earliest_empty("28"), 2, 'earliest_empty("28")')
check(latest_empty("28"), 8, 'latest_empty("28")')
check(earliest_empty("15"), 5, 'earliest_empty("15")')
check(latest_empty("15"), 9, 'latest_empty("15")')
check(earliest_empty("123456789"), 5, 'earliest_empty("123456789")')
check(latest_empty("123456789"), 9, 'latest_empty("123456789")')
check(earliest_empty(""), 0, 'earliest_empty("")')
check(latest_empty(""), 0, 'latest_empty("")')

## Stage 6: Evaluate (about 10 minutes)


Your functions are fast, but are they **right**? The simulator gives us a second, completely different way to get the answer: try every possible choice of directions and simulate each one (*brute force*). It is slow, but it follows the rules of the problem exactly, collisions and all, so it makes a good *test oracle*. `sim_earliest(positions)` and `sim_latest(positions)` return its answers.

Add some inputs of your own to the list of tests in the cell below, and run it.

In [ ]:
# 6.1 Compare your functions with the brute-force simulator.
# Add your own test inputs to this string, separated by spaces (digits 1-9, no repeats in one input).
tests = "5 19 28 37 46 1234 2468 13579 123456789"

current = ""
for ch in tests + " ":
    if ch == " ":
        if current != "":
            check(earliest_empty(current), sim_earliest(current), f"earliest_empty({current!r})")
            check(latest_empty(current), sim_latest(current), f"latest_empty({current!r})")
        current = ""
    else:
        current = current + ch

### Check your evaluation

1. Did your functions agree with the simulator on every input? If not, which inputs disagreed, and what was wrong?
2. `simulate_all` simulates every combination of directions. How many simulations does it run for 2 ants? For 9 ants? Roughly how many would it need for 100 ants on a 1 m stick?
3. How many steps does *your* function need for 100 ants?
4. Why do we not need to simulate collisions to get the answer?
5. The simulator only allows positions 1 to 9 and a 10 cm stick. Which parts of your functions would change for a stick of a different length?

✏️ **Your answers** (double-click to edit):

1. 
2. 
3. 
4. 
5. 

### Extension ★: generalize

In [ ]:
# 6.2 Extension: any length of stick
# Copy your functions here, and use the new parameter length instead of 10.
def earliest_empty(positions, length=10):
    pass


def latest_empty(positions, length=10):
    pass

# ---- Checks: run the cell to test your code (do not change below) ----
check(earliest_empty("28", 12), 4, 'earliest_empty("28", 12)')
check(latest_empty("28", 12), 10, 'latest_empty("28", 12)')
check(earliest_empty("28"), 2, 'earliest_empty("28")')

## Before you leave

Tick off each statement you can honestly agree with. For any you can't, go back to the matching part of this lab, the lecture workbook, or the book chapter.

- [ ] I can find what is ambiguous in a problem statement and ask questions to pin it down.
- [ ] I can explain why collisions do not change when the stick becomes empty.
- [ ] I can explain why the earliest time is the *largest* nearer-end distance, not the smallest.
- [ ] I can test a fast solution by comparing it with a slow brute-force one.

✏️ **Reflection** (double-click to edit; answer together with your partner):

1. Which computational-thinking skill helped most with this problem: abstraction, decomposition, pattern recognition, algorithm design, or validation? Why?
2. At what point did you change your understanding of the problem? What made you change it?
3. Describe another problem where a slow but simple solution could be used to test a fast one.

*Your answers:*

1. 
2. 
3. 